# Interactive Graphics with Plotly & Dash

**Plotly** is an open-source graphing library for interactive, web-based visualizations.
**Dash** is a low-code framework for building full data apps in Python, written on top of Plotly.js and React.js.

Since Plotly 6 / Dash 3 (2025), Plotly Express works natively with pandas, Polars, and PyArrow DataFrames (via Narwhals).

**Documentation:**
- Plotly: https://plotly.com/python/
- Dash: https://dash.plotly.com

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Load apartment data
df = pd.read_csv('../../data/apartments_data_enriched_cleaned.csv', sep=';', encoding='utf-8')
print('Shape:', df.shape)
df.head(3)

---
## Plotly Express — High-Level Interface

### Histogram

In [ ]:
fig = px.histogram(
    df, x='price', nbins=50,
    title='Distribution of Apartment Prices',
    labels={'price': 'Price (CHF)'},
    template='plotly_white'
)
fig.show()

### Box Plot

In [ ]:
fig = px.box(
    df, x='rooms', y='price', color='luxurious',
    title='Apartment Price by Rooms and Luxurious Category',
    labels={'price': 'Price (CHF)', 'rooms': 'Rooms'},
    template='plotly_white'
)
fig.show()

### Scatter Plot

In [ ]:
sample = df.dropna(subset=['price', 'area', 'rooms']).sample(500, random_state=42)

fig = px.scatter(
    sample, x='area', y='price',
    color='luxurious', size='rooms',
    hover_data=['rooms'],
    title='Price vs. Area (color=luxurious, size=rooms)',
    labels={'area': 'Area (m²)', 'price': 'Price (CHF)'},
    template='plotly_white',
    opacity=0.6
)
fig.show()

### Scatter Plot Matrix (SPLOM)

In [ ]:
num_cols = ['price', 'area', 'rooms']
fig = px.scatter_matrix(
    df[num_cols + ['luxurious']].dropna().sample(500, random_state=42),
    dimensions=num_cols,
    color='luxurious',
    title='Scatter Plot Matrix — Apartment Data',
    template='plotly_white',
    opacity=0.4
)
fig.update_traces(diagonal_visible=False)
fig.show()

### Correlation Heatmap

In [ ]:
corr = df.corr(method='pearson', numeric_only=True).round(2)

fig = px.imshow(
    corr, text_auto=True, color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Pearson Correlation Heatmap — Apartment Data',
    template='plotly_white'
)
fig.show()

---
## Dash — Building a Data App

Dash apps run as web servers. The example below creates a simple interactive histogram where the user can select the variable and number of bins.

To run: save as `app.py` and execute `python app.py`, then open http://127.0.0.1:8050 in your browser.

In [ ]:
# Dash app source code (for reference — run as a standalone script)
dash_app_code = '''
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

df = pd.read_csv('../../data/apartments_data_enriched_cleaned.csv', sep=';', encoding='utf-8')
num_cols = ['price', 'area', 'rooms']

app = Dash(__name__)
app.layout = html.Div([
    html.H2('Apartment Data EDA'),
    dcc.Dropdown(id='variable', options=[{'label': c, 'value': c} for c in num_cols],
                 value='price', clearable=False, style={'width': '300px'}),
    dcc.Slider(id='bins', min=10, max=100, step=5, value=40,
               marks={i: str(i) for i in range(10, 101, 10)}),
    dcc.Graph(id='histogram')
])

@app.callback(
    Output('histogram', 'figure'),
    Input('variable', 'value'),
    Input('bins', 'value')
)
def update_histogram(variable, bins):
    return px.histogram(df, x=variable, nbins=bins,
                        title=f'Distribution of {variable}', template='plotly_white')

if __name__ == '__main__':
    app.run(debug=True)
'''
print(dash_app_code)

## Jupyter notebook --footer info--
(please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('------------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('------------------------------------')